# 03. Behavior Cloning 学習 (Google Colab)

ローカルで収集した `data/episodes/`(画像フォルダ群)と `data/manifest.csv` を使って、
模倣学習 (Behavior Cloning) でCNNを学習します。このノートブックは **Google Colab** にアップロードして実行してください。

## 事前準備 (ローカルで実行)

PowerShellで `data` フォルダをzip化し、Googleドライブの適当なフォルダ (例: `MyDrive/mario_bc/`) にアップロードしてください。
画像ファイルが大量になるため、個別アップロードではなく zip化してからアップロードすること。

```powershell
Compress-Archive -Path data -DestinationPath mario_data.zip
```

## Colabでの実行手順

1. 「ランタイム」→「ランタイムのタイプを変更」→ ハードウェアアクセラレータを **GPU** に設定
2. 以下のセルを上から順に実行(まずは2〜3エピソードだけで1epochの試験実行をしてエラーが出ないことを確認してから、本学習に進むことを推奨)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import zipfile
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/mario_bc/mario_data.zip"  # 実際のアップロード先に合わせて変更してください
EXTRACT_DIR = Path("/content/mario_data")

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(EXTRACT_DIR)

DATA_DIR = EXTRACT_DIR / "data"
EPISODES_DIR = DATA_DIR / "episodes"
MANIFEST_PATH = DATA_DIR / "manifest.csv"

episode_dirs_preview = sorted(EPISODES_DIR.glob("ep_*"))
print(f"{len(episode_dirs_preview)} episodes found")
print(episode_dirs_preview[:5])

In [ ]:
# gym_super_mario_bros==9.1.0 の SIMPLE_MOVEMENT をそのまま複製したもの。
# Colab側では実機プレイ用の nes-py 一式は不要なため、行動リストの定義だけをここに持つ。
# ローカルの gym_super_mario_bros.actions.SIMPLE_MOVEMENT (v9.1.0) と内容が一致していることを前提にしている。
SIMPLE_MOVEMENT = [
    ["NOOP"],
    ["right"],
    ["right", "A"],
    ["right", "B"],
    ["right", "A", "B"],
    ["A"],
    ["left"],
]
print(SIMPLE_MOVEMENT)

In [ ]:
# --- この定義は notebooks/03_train_bc.ipynb と notebooks/04_playagent.ipynb で完全に一致させること ---
from collections import deque

import cv2
import numpy as np
import torch
import torch.nn as nn

FRAME_SIZE = (84, 84)
STACK_SIZE = 4
NUM_ACTIONS = len(SIMPLE_MOVEMENT)


def preprocess_frame(frame: np.ndarray) -> np.ndarray:
    """RGBフレームをグレースケール化し 84x84 にリサイズする"""
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    return cv2.resize(gray, FRAME_SIZE, interpolation=cv2.INTER_AREA)


class FrameStacker:
    """前処理済みフレームを直近 STACK_SIZE 枚スタックする"""

    def __init__(self, stack_size: int = STACK_SIZE):
        self.stack_size = stack_size
        self.frames = deque(maxlen=stack_size)

    def reset(self, processed_frame: np.ndarray) -> np.ndarray:
        self.frames.clear()
        for _ in range(self.stack_size):
            self.frames.append(processed_frame)
        return self.get()

    def step(self, processed_frame: np.ndarray) -> np.ndarray:
        self.frames.append(processed_frame)
        return self.get()

    def get(self) -> np.ndarray:
        return np.stack(self.frames, axis=0)


class MarioCNN(nn.Module):
    def __init__(self, num_actions: int = NUM_ACTIONS, in_channels: int = STACK_SIZE):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=8, stride=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(inplace=True),
            nn.Flatten(),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, *FRAME_SIZE)
            n_flatten = self.features(dummy).shape[1]
        self.head = nn.Sequential(
            nn.Linear(n_flatten, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.float() / 255.0
        x = self.features(x)
        return self.head(x)
# --- ここまで ---

In [ ]:
import re

from PIL import Image

FILENAME_RE = re.compile(r"^(\d+)_a(\d+)_")


def list_episode_dirs(episodes_dir: Path):
    return sorted(p for p in episodes_dir.glob("ep_*") if p.is_dir())


def load_episode(ep_dir: Path):
    """エピソードフォルダの画像を読み込み、前処理済みフレームとアクションのリストを返す"""
    image_paths = sorted(ep_dir.glob("*.png"))
    frames, action_list = [], []
    for path in image_paths:
        m = FILENAME_RE.match(path.name)
        if not m:
            continue
        action_list.append(int(m.group(2)))
        with Image.open(path) as img:
            frames.append(preprocess_frame(np.array(img.convert("RGB"))))
    return frames, action_list


class MarioEpisodeDataset(torch.utils.data.Dataset):
    def __init__(self, episode_dirs):
        self.stacks = []
        self.labels = []
        for ep_dir in episode_dirs:
            frames, action_list = load_episode(ep_dir)
            if not frames:
                continue
            stacker = FrameStacker()
            self.stacks.append(stacker.reset(frames[0]))
            self.labels.append(action_list[0])
            for frame, action in zip(frames[1:], action_list[1:]):
                self.stacks.append(stacker.step(frame))
                self.labels.append(action)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        stack = torch.from_numpy(self.stacks[idx].copy())
        return stack, self.labels[idx]

## train/val 分割 (エピソード単位)

フレーム単位で分割すると隣接フレームがtrain/valに漏れて検証精度が過大評価されるため、エピソード単位で分割する。

In [ ]:
import random

episode_dirs = list_episode_dirs(EPISODES_DIR)
random.seed(0)
shuffled = episode_dirs[:]
random.shuffle(shuffled)

split_idx = max(1, int(len(shuffled) * 0.85))
train_dirs = shuffled[:split_idx]
val_dirs = shuffled[split_idx:] or shuffled[-1:]

print(f"train episodes: {len(train_dirs)}, val episodes: {len(val_dirs)}")

train_dataset = MarioEpisodeDataset(train_dirs)
val_dataset = MarioEpisodeDataset(val_dirs)
print(f"train samples: {len(train_dataset)}, val samples: {len(val_dataset)}")

## クラス重みの計算

`02_review_data.ipynb` で確認したように "right" 等の多数派アクションにサンプルが偏っているため、出現頻度の逆数でクラス重み付けする。

In [ ]:
from collections import Counter

label_counts = Counter(train_dataset.labels)
class_weights = torch.tensor(
    [1.0 / label_counts.get(i, 1) for i in range(NUM_ACTIONS)],
    dtype=torch.float32,
)
class_weights = class_weights / class_weights.sum() * NUM_ACTIONS

print("クラスごとのサンプル数:", dict(sorted(label_counts.items())))
print("クラス重み:", class_weights.tolist())

## 学習

動作確認は `EPOCHS = 1` かつ少数エピソードで、本学習は `EPOCHS = 15` 程度から調整してください。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

EPOCHS = 15
BATCH_SIZE = 64
LEARNING_RATE = 2.5e-4

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = MarioCNN().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for stacks, labels in train_loader:
        stacks, labels = stacks.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(stacks)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    train_loss = total_loss / total
    train_acc = correct / total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for stacks, labels in val_loader:
            stacks, labels = stacks.to(device), labels.to(device)
            logits = model(stacks)
            val_correct += (logits.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total if val_total else float("nan")

    print(
        f"epoch {epoch:3d}/{EPOCHS}  train_loss={train_loss:.4f}  "
        f"train_acc={train_acc:.3f}  val_acc={val_acc:.3f}"
    )

## 混同行列 (検証データ)

7クラス中の多数派アクションに引っ張られて全体精度だけでは判断しにくいため、クラスごとの予測傾向も確認する。

In [ ]:
import pandas as pd

confusion = np.zeros((NUM_ACTIONS, NUM_ACTIONS), dtype=int)
model.eval()
with torch.no_grad():
    for stacks, labels in val_loader:
        preds = model(stacks.to(device)).argmax(dim=1).cpu().numpy()
        for t, p in zip(labels.numpy(), preds):
            confusion[t, p] += 1

action_names = ["_".join(a) if a else "NOOP" for a in SIMPLE_MOVEMENT]
pd.DataFrame(confusion, index=action_names, columns=action_names)

## モデルの書き出し

In [ ]:
import time as _time

from google.colab import files

CHECKPOINT_NAME = f"mario_bc_{_time.strftime('%Y%m%d_%H%M%S')}.pt"
torch.save(model.state_dict(), CHECKPOINT_NAME)
print("saved:", CHECKPOINT_NAME)

files.download(CHECKPOINT_NAME)

ダウンロードした `.pt` ファイルをローカルの `models/checkpoints/` に配置してください。`04_playagent.ipynb` で使用します。